# Constrained Optimization — Maximize X with limits on Y and Z

**Goal:** find the experiment(s) that maximise **X** while keeping **Y ≤ limit_Y** and **Z ≤ limit_Z**.

**Parameters:** A, B, C, D, E, F

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Configuration

In [ ]:
# ── Edit these ───────────────────────────────────────────────────────────────
CSV_PATH = 'your_data.csv'

LIMIT_Y = 10.0   # keep Y at or below this value
LIMIT_Z = 5.0    # keep Z at or below this value
# ─────────────────────────────────────────────────────────────────────────────

PARAMS  = ['A', 'B', 'C', 'D', 'E', 'F']
ALL_OBJ = ['X', 'Y', 'Z']

## 2. Load data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} rows, columns: {list(df.columns)}')
df.head()

## 3. Constraint diagnostics

Check how many rows survive each constraint before combining them.

In [ ]:
mask_y = df['Y'] <= LIMIT_Y
mask_z = df['Z'] <= LIMIT_Z
mask_both = mask_y & mask_z

print(f'Y ≤ {LIMIT_Y}          : {mask_y.sum():>5} / {len(df)} rows ({100*mask_y.mean():.1f}%)')
print(f'Z ≤ {LIMIT_Z}           : {mask_z.sum():>5} / {len(df)} rows ({100*mask_z.mean():.1f}%)')
print(f'Both constraints   : {mask_both.sum():>5} / {len(df)} rows ({100*mask_both.mean():.1f}%)')

if mask_both.sum() == 0:
    print('\n⚠  No feasible experiments — consider relaxing LIMIT_Y or LIMIT_Z.')
    print(f'   Y range: [{df["Y"].min():.3g}, {df["Y"].max():.3g}]')
    print(f'   Z range: [{df["Z"].min():.3g}, {df["Z"].max():.3g}]')

## 4. Apply constraints and find the best experiment

In [ ]:
feasible = df[mask_both].copy()

if feasible.empty:
    raise ValueError('No feasible rows — relax your limits in section 1.')

best_idx = feasible['X'].idxmax()
best     = feasible.loc[best_idx]

print('Best experiment:')
print(best[PARAMS + ALL_OBJ].to_string())

# Top-10 feasible experiments sorted by X
print('\nTop-10 feasible experiments (sorted by X):')
feasible.sort_values('X', ascending=False)[PARAMS + ALL_OBJ].head(10)

## 5. Visualise constraints and best point

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (ycol, limit, label) in zip(
    axes,
    [('Y', LIMIT_Y, f'Y ≤ {LIMIT_Y}'), ('Z', LIMIT_Z, f'Z ≤ {LIMIT_Z}')]
):
    infeasible = df[~mask_both]
    ax.scatter(infeasible['X'], infeasible[ycol],
               alpha=0.2, s=15, color='grey', label='Infeasible')
    ax.scatter(feasible['X'], feasible[ycol],
               alpha=0.5, s=20, color='steelblue', label='Feasible')
    ax.scatter(best['X'], best[ycol],
               color='red', s=100, zorder=6, label='Best', marker='*')
    ax.axhline(limit, color='orange', linewidth=1.5, linestyle='--', label=f'Limit ({limit})')
    ax.set_xlabel('X (maximise)')
    ax.set_ylabel(ycol)
    ax.set_title(f'X vs {ycol}  |  constraint: {label}')
    ax.legend(fontsize=8)

plt.suptitle('Constrained optimisation — feasible region', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Distribution of objectives — all vs feasible

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))

for ax, col in zip(axes, ALL_OBJ):
    ax.hist(df[col],       bins=30, alpha=0.4, color='grey',      label='All')
    ax.hist(feasible[col], bins=30, alpha=0.7, color='steelblue', label='Feasible')
    if col == 'Y':
        ax.axvline(LIMIT_Y, color='orange', linestyle='--', label=f'Limit {LIMIT_Y}')
    if col == 'Z':
        ax.axvline(LIMIT_Z, color='orange', linestyle='--', label=f'Limit {LIMIT_Z}')
    if col == 'X':
        ax.axvline(best['X'], color='red', linestyle='--', label=f'Best {best["X"]:.3g}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Objective distributions', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Sensitivity — how does the feasible set size change with limits?

In [ ]:
y_range = np.linspace(df['Y'].min(), df['Y'].max(), 40)
z_range = np.linspace(df['Z'].min(), df['Z'].max(), 40)

grid = np.array([[(df['Y'] <= y) & (df['Z'] <= z)]
                  for y in y_range for z in z_range])
counts = grid.reshape(len(y_range), len(z_range), len(df)).sum(axis=2)

# Best X achievable for each (limit_Y, limit_Z) pair
best_x = np.full((len(y_range), len(z_range)), np.nan)
for i, y in enumerate(y_range):
    for j, z in enumerate(z_range):
        sub = df[(df['Y'] <= y) & (df['Z'] <= z)]
        if not sub.empty:
            best_x[i, j] = sub['X'].max()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].contourf(z_range, y_range, counts, levels=20, cmap='Blues')
axes[0].axhline(LIMIT_Y, color='red', linestyle='--', label='Current LIMIT_Y')
axes[0].axvline(LIMIT_Z, color='orange', linestyle='--', label='Current LIMIT_Z')
axes[0].set_xlabel('LIMIT_Z'); axes[0].set_ylabel('LIMIT_Y')
axes[0].set_title('Feasible row count')
axes[0].legend(fontsize=8)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].contourf(z_range, y_range, best_x, levels=20, cmap='RdYlGn')
axes[1].axhline(LIMIT_Y, color='red', linestyle='--', label='Current LIMIT_Y')
axes[1].axvline(LIMIT_Z, color='orange', linestyle='--', label='Current LIMIT_Z')
axes[1].set_xlabel('LIMIT_Z'); axes[1].set_ylabel('LIMIT_Y')
axes[1].set_title('Best achievable X')
axes[1].legend(fontsize=8)
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Sensitivity to constraint limits', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Export feasible experiments

In [ ]:
out_path = CSV_PATH.replace('.csv', '_feasible.csv')
feasible.sort_values('X', ascending=False)[PARAMS + ALL_OBJ].to_csv(out_path, index=False)
print(f'Saved {len(feasible)} feasible rows to {out_path}')